### Test the [Smol-VLM from Huggingface](https://huggingface.co/blog/smolvlm)

In [13]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.set_num_threads(torch.get_num_threads())  # Use all CPU cores
torch.backends.mkldnn.enabled = True  # Enable optimized matrix operations

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
model = AutoModelForVision2Seq.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                torch_dtype=torch.bfloat16,
                                                _attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager").to(DEVICE)


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


In [5]:
from PIL import Image
from transformers.image_utils import load_image


# Load images
image = load_image("https://huggingface.co/spaces/HuggingFaceTB/SmolVLM/resolve/main/example_images/rococo.jpg")

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Can you describe the image?"}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=image, return_tensors="pt")
inputs = inputs.to(DEVICE)


In [3]:
from pdf_utilities import load_pdf, save_images, text_extraction

pdf = "WhatIsOMERO.pdf"
slides = load_pdf(pdf)

In [10]:
from PIL import Image
from transformers.image_utils import load_image


# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Give me a short and precise summary (1-3 whole sentences) of what is displayed or explained in this Slide."}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=slides[0], return_tensors="pt")
inputs = inputs.to(DEVICE)


In [16]:
import time

start = time.time()
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=100)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)
end = time.time()

print(generated_texts[0])
totaltime = end - start
print(f'Duration: {totaltime} seconds')

User:<image>Give me a short and precise summary (1-3 whole sentences) of what is displayed or explained in this Slide.
Assistant: This slide provides instructions on how to use the slide templates for OMERO user training slides. It mentions that the slides are optimized for a 16:9 screen presentation layout and should be checked for yellow-marked text. It also advises to insert the information according to the user's institute's infrastructure.
Duration: 141.28266882896423 seconds


In [4]:
import time
from endpoints import prompt_chatgpt

start = time.time()
result = prompt_chatgpt(f"{pdf}_slide1.png", "Give me a short and precise summary (1-3 whole sentences) of what is displayed or explained in this Slide.")
print(result)
end = time.time()
totaltime = end - start
print(f'Duration: {totaltime} seconds')

This slide provides instructions for customizing and using the provided training slide templates, optimized for a 16:9 layout. It highlights the need to update yellow-marked text with specific information and outlines usage guidelines under a Creative Commons license.
Duration: 2.6155011653900146 seconds
